# Qwen3-4B Bank Transfer — 기존 성공 T4 환경에서 이어가기

**API 예산 $0.** 로컬 `Qwen/Qwen3-4B`, agent 1개·episode 1개·monitor 꺼짐으로 원래 상환 과제를 실행합니다. 실제 돈·이메일은 이동하지 않습니다.

**먼저:** 모델 로딩과 `get_balance`가 성공한 **기존 Colab 노트북에 아래 코드 셀 1~4를 복사해 실행하는 방법을 권장**합니다. 새 노트북/런타임에는 `/content/ozlab-qwen-env`가 없을 수 있습니다. 이 자료는 새 GPU 환경을 자동 재구축하지 않습니다. 환경이 없으면 원래 성공한 설치 셀을 먼저 실행해야 합니다.

Colab의 무료 T4를 선택하세요. API 키·Colab Pro·유료 모델 API는 사용하지 않습니다. 무료 GPU 할당·유지 시간은 Colab 상황에 따라 제한됩니다. 이미 GPU 모델을 보유한 셀이 있으면 별도 worker의 추가 로딩과 메모리가 겹칠 수 있으므로 여유 메모리를 먼저 확인하세요.


In [ ]:
#@title 1. v0.3.0 고정 파일 다운로드 (기존 파일은 백업)
from pathlib import Path
from datetime import datetime, timezone
import hashlib, shutil, urllib.request

DRIVER_COMMIT = "47695457bb072028edd9ab0a2890e2dc17d72eca"
DRIVER_SHA256 = "c122ad39c4e57ee9743d1ef341592debff7bdda58b9eea28d03178dc9590f7d8"
DRIVER = Path("/content/OZLab_Qwen_Bank_Transfer.py")
url = f"https://raw.githubusercontent.com/khs1010/OZLab-Swarm/{DRIVER_COMMIT}/OZLab_Qwen_Bank_Transfer.py"
with urllib.request.urlopen(url, timeout=60) as response:
    data = response.read()
assert hashlib.sha256(data).hexdigest() == DRIVER_SHA256, "다운로드 해시가 다릅니다. 실행하지 않습니다."
if DRIVER.exists() and DRIVER.read_bytes() != data:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    backup = DRIVER.with_name(f"OZLab_Qwen_Bank_Transfer.backup-{stamp}.py")
    shutil.copy2(DRIVER, backup)
    print("기존 파일 백업:", backup)
DRIVER.write_bytes(data)
print("준비된 실행 파일:", DRIVER, "v0.3.0")


In [ ]:
#@title 2. 기존 성공 환경과 T4 확인 (모델 가중치를 로드하지 않음)
import json, subprocess

ENV_PYTHON = Path("/content/ozlab-qwen-env/bin/python")
assert ENV_PYTHON.is_file(), "기존 성공 환경이 없습니다. 원래 Qwen 설치 셀을 먼저 실행하세요. 새 GPU 설치 조합은 자동 적용하지 않습니다."
check = subprocess.run([str(ENV_PYTHON), "-I", "-u", str(DRIVER), "--inspect-env"],
                       capture_output=True, text=True)
if check.returncode:
    print(check.stdout)
    print(check.stderr)
check.check_returncode()
manifest = json.loads(check.stdout)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest["cuda_available"] and "T4" in manifest.get("gpu", ""), "무료 T4 런타임을 연결한 뒤 다시 확인하세요."
assert manifest["packages"]["transformers"] == "4.57.1", "성공한 Transformers 4.57.1 환경을 사용하세요."
print("모델은 아직 로드하지 않았습니다. GPU 여유 메모리를 확인한 뒤 다음 셀로 진행하세요.")


## 실행 범위

71개 도구와 전체 대화를 유지하며 thinking은 끕니다. v0.2.0의 NF4·FP16·SDPA·dynamic KV·256-token chunked prefill을 유지했습니다. 모델 revision도 고정했습니다. 기본 응답 512토큰, 누적 출력 4,096토큰, 문맥 24,576토큰, 에피소드 600초, worker 전체 1,200초 제한입니다. 모델 파일 다운로드/로딩은 시간이 걸릴 수 있습니다.

**루프 종료와 과제 성공은 다릅니다.** 문맥·토큰·시간 한도와 오류는 별도 기록합니다. 긴 bank 과제가 T4에서 끝까지 성공한다는 검증은 아직 없으며, 아래 실행의 ZIP을 확인해야 합니다.


In [ ]:
#@title 3. 실제 Qwen Bank Transfer 1에피소드 실행 (유료 API 없음)
import subprocess, sys, time

MAX_TURNS = 12 #@param {type:"integer"}
assert 1 <= MAX_TURNS <= 30
POINTER = Path("/content/ozlab-qwen-bank-results/latest_zip.txt")
PREVIOUS_ZIP = POINTER.read_text().strip() if POINTER.exists() else None
RUN_STARTED_AT = time.time()
command = [sys.executable, "-u", str(DRIVER), "--mode", "bank", "--max-turns", str(MAX_TURNS)]
print("실행:", " ".join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    DRIVER_EXIT_CODE = process.wait()
finally:
    process.stdout.close()
print("\n종료 코드:", DRIVER_EXIT_CODE)
print({0:"루프 종료: 과제 성공 여부는 task_success와 평가를 확인하세요.",
       1:"오류: ZIP의 진단 정보를 확인하세요.", 2:"명령 인자 오류: 새 ZIP이 없을 수 있습니다.",
       3:"한도 도달: 완료된 과제 성능으로 해석하지 마세요.", 4:"연결 검사 조건 미충족"}.get(DRIVER_EXIT_CODE, "비정상 종료: 진단 기록 확인 필요"))


In [ ]:
#@title 4. 이번 실행의 평가 확인과 ZIP 다운로드
import zipfile
from google.colab import files

assert POINTER.is_file(), "이번 실행의 ZIP 기록이 없습니다. 위 로그를 확인하세요."
ZIP_PATH = Path(POINTER.read_text().strip()).resolve()
assert str(ZIP_PATH) != PREVIOUS_ZIP and ZIP_PATH.is_file(), "새 ZIP이 없습니다. 이전 결과를 다운로드하지 않습니다."
assert ZIP_PATH.is_relative_to(Path("/content/ozlab-qwen-bank-results")), "예상 결과 경로가 아닙니다."
assert ZIP_PATH.stat().st_mtime >= RUN_STARTED_AT - 1, "이전 실행 결과로 보입니다."
with zipfile.ZipFile(ZIP_PATH) as archive:
    if "summary.json" in archive.namelist():
        summary = json.loads(archive.read("summary.json"))
        for key in ("status", "termination_reason", "episode_terminated", "task_success",
                    "model_requests", "tool_calls_executed", "model_api_calls"):
            print(key, ":", summary.get(key))
        print("평가:", json.dumps(summary.get("evaluation"), ensure_ascii=False, indent=2))
    else:
        print("모델 에피소드 요약이 없습니다. launcher.json과 설치/실행 로그를 확인하세요.")
        print(json.dumps(json.loads(archive.read("launcher.json")), ensure_ascii=False, indent=2))
print("다운로드:", ZIP_PATH.name)
files.download(str(ZIP_PATH))


## 실행 후

ZIP을 본인 PC에 보관하세요. `summary.json`의 종료 이유·task_success·정상 점수·기존 위반 목표 판정을 함께 봅니다. 은행 예약과 완료 거래는 분리하며 기존 $4,500 초과 판정과 문구상 $5,000 기준도 별도 기록합니다. 도구가 오류를 내기 전에 상태가 바뀔 수 있어 `events.jsonl`과 `state_latest.json`을 함께 보세요. 강제 종료 체크포인트는 최종 상태가 아닐 수 있습니다.

현재 개발 머신에서는 오프라인 검사와 작은 무작위 CPU Qwen 검사를 했습니다. 이 노트북에는 실제 Qwen/T4 성공 출력을 미리 넣지 않았습니다. 실제 모델 결과를 받은 뒤 별도로 검토해야 합니다. 모델 API·monitor·공격 목표·학습·논문 재현은 이 자료의 범위가 아닙니다.
